# 02 Train Emulator (PINN from scratch)

Addestramento PINN indipendente dalla MLP: usa solo il dataset condiviso.


In [1]:
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
_candidates = [_cwd, _cwd / 'PINN', _cwd.parent, _cwd.parent / 'PINN', _cwd.parent.parent]
_PROJECT_ROOT = next((p for p in _candidates if (p / 'src' / 'solsys_emulator').exists()), None)
if _PROJECT_ROOT is None:
    raise RuntimeError('Impossibile trovare la project root con src/solsys_emulator')

_SRC = _PROJECT_ROOT / 'src'
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

print('Project root:', _PROJECT_ROOT)
print('Python executable:', sys.executable)



Project root: /Users/francescosulli/Documents/ASTREO/IA e simulazioni/solar_system/PINN
Python executable: /Applications/Xcode.app/Contents/Developer/usr/bin/python3


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import torch

from solsys_emulator.config import DEFAULT_CHECKPOINT_PATH, DEFAULT_DATASET_PATH
from solsys_emulator.de440_dataset import load_dataset
from solsys_emulator.model import ModelConfig
from solsys_emulator.train import TrainConfig, train_emulator

dataset = load_dataset(DEFAULT_DATASET_PATH)
print('Dataset source:', dataset.get('metadata', {}).get('sample_source'))
print('States shape:', dataset['states'].shape)
print('Num samples:', len(dataset['times_seconds']))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_gpu_profile = device == 'cuda'
loader_workers = 8 if use_gpu_profile else 0
pin_memory = True if use_gpu_profile else None
persistent_workers = True if use_gpu_profile and loader_workers > 0 else False

print('Device:', device)
print('GPU-heavy profile:', use_gpu_profile)

pinn_ckpt = Path(DEFAULT_CHECKPOINT_PATH)

if use_gpu_profile:
    model_cfg = ModelConfig(
        num_bodies=len(dataset['bodies']),
        state_mode='position_only',
        backbone_type='residual',
        hidden_dim=640,
        num_layers=8,
        fourier_features=80,
        min_frequency=0.02,
        max_frequency=96.0,
        frequency_spacing='log',
        head_layers=3,
        head_hidden_dim=320,
        body_embedding_dim=64,
        interaction_layers=2,
        interaction_hidden_dim=256,
        use_layer_norm=True,
        dropout=0.0,
    )
    train_cfg = TrainConfig(
        epochs=1400,
        batch_size=768,
        gradient_accumulation_steps=2,
        lr=2.5e-4,
        weight_decay=5e-7,
        val_fraction=0.10,
        split_mode='random',
        shuffle=True,
        train_loader_workers=loader_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        early_stopping_patience=260,
        lr_scheduler='cosine',
        min_lr=5e-7,
        nbody_loss_weight=4e-6,
        adaptive_nbody_balance=True,
        nbody_target_fraction=1e-2,
        nbody_balance_beta=0.97,
        nbody_balance_max_scale=1e8,
        nbody_collocation_points=192,
        nbody_start_epoch=40,
        nbody_warmup_epochs=320,
        nbody_softening_km=50_000.0,
        nbody_relative_floor_km_s2=2e-4,
        physics_loss_weight=0.0,
        smoothness_loss_weight=0.0,
        energy_loss_weight=2e-4,
        angular_momentum_loss_weight=2e-4,
        energy_start_epoch=40,
        angular_momentum_start_epoch=40,
        energy_warmup_epochs=320,
        angular_momentum_warmup_epochs=320,
        position_loss_weight=1.0,
        velocity_loss_weight=0.5,
        grad_clip_norm=1.0,
        compute_val_velocity_rmse=True,
        selection_metric='val_pos_rmse_km',
        force_chronological_for_derivatives=False,
        sort_train_for_derivatives=False,
        show_progress=True,
        device=device,
        cuda_matmul_precision='high',
        allow_tf32=True,
        cudnn_benchmark=True,
    )
else:
    model_cfg = ModelConfig(
        num_bodies=len(dataset['bodies']),
        state_mode='position_only',
        backbone_type='plain',
        hidden_dim=384,
        num_layers=6,
        fourier_features=40,
        min_frequency=0.25,
        max_frequency=48.0,
        frequency_spacing='log',
        head_layers=2,
        head_hidden_dim=160,
        body_embedding_dim=0,
        interaction_layers=0,
        interaction_hidden_dim=128,
        use_layer_norm=False,
        dropout=0.0,
    )
    train_cfg = TrainConfig(
        epochs=600,
        batch_size=384,
        lr=2e-4,
        weight_decay=1e-6,
        val_fraction=0.10,
        split_mode='random',
        shuffle=True,
        train_loader_workers=loader_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        early_stopping_patience=120,
        lr_scheduler='cosine',
        min_lr=5e-7,
        nbody_loss_weight=1.2e-6,
        adaptive_nbody_balance=True,
        nbody_target_fraction=3e-3,
        nbody_balance_beta=0.95,
        nbody_balance_max_scale=1e6,
        nbody_collocation_points=64,
        nbody_start_epoch=20,
        nbody_warmup_epochs=140,
        nbody_softening_km=80_000.0,
        nbody_relative_floor_km_s2=5e-4,
        physics_loss_weight=0.0,
        smoothness_loss_weight=0.0,
        energy_loss_weight=5e-5,
        angular_momentum_loss_weight=5e-5,
        energy_start_epoch=20,
        angular_momentum_start_epoch=20,
        energy_warmup_epochs=140,
        angular_momentum_warmup_epochs=140,
        position_loss_weight=1.0,
        velocity_loss_weight=0.35,
        grad_clip_norm=1.0,
        compute_val_velocity_rmse=True,
        selection_metric='val_pos_rmse_km',
        force_chronological_for_derivatives=False,
        sort_train_for_derivatives=False,
        show_progress=True,
        device=device,
    )

print('Model config:', model_cfg)
print('Train config:', train_cfg)
print('Note: in position_only PINN the active physics terms are n-body residual plus conservation regularizers; legacy physics/smoothness losses remain disabled by design.')

train_artifacts = train_emulator(
    dataset,
    train_config=train_cfg,
    model_config=model_cfg,
    checkpoint_path=pinn_ckpt,
)

best_epoch = int(np.argmin(train_artifacts['history']['val_pos_rmse_km'])) + 1
best_rmse = float(np.min(train_artifacts['history']['val_pos_rmse_km']))

print('Final PINN checkpoint:', pinn_ckpt)
print('Best epoch:', best_epoch)
print('Best val position RMSE [km]:', f'{best_rmse:,.2f}')

history = train_artifacts['history']

plt.figure(figsize=(10, 4))
plt.plot(history['train_loss'], label='train data loss')
if 'train_objective_loss' in history:
    plt.plot(history['train_objective_loss'], label='train objective loss')
plt.plot(history['val_loss'], label='val data loss')
plt.yscale('log')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('PINN unified training history')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 3))
plt.plot(history['val_pos_rmse_km'], label='val position RMSE [km]')
plt.plot(history['val_vel_rmse_km_s'], label='val velocity RMSE [km/s]')
plt.yscale('log')
plt.xlabel('epoch')
plt.ylabel('RMSE')
plt.title('PINN unified validation RMSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 3))
nbody_raw = np.array(history['nbody_loss'], dtype=float)
nbody_w = np.array(history['nbody_weight'], dtype=float)
plt.plot(np.maximum(1e-16, nbody_raw), label='nbody raw')
plt.plot(np.maximum(1e-16, nbody_raw * nbody_w), label='nbody weighted')
plt.yscale('log')
plt.xlabel('epoch')
plt.ylabel('loss contribution scale')
plt.title('PINN n-body residual diagnostics')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(10, 3))
plt.plot(np.maximum(1e-16, np.array(history['energy_loss'], dtype=float)), label='energy raw')
plt.plot(np.maximum(1e-16, np.array(history['angular_momentum_loss'], dtype=float)), label='angular momentum raw')
plt.yscale('log')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('PINN conservation diagnostics')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 3))
plt.plot(history['lr'])
plt.xlabel('epoch')
plt.ylabel('lr')
plt.title('PINN learning-rate schedule')
plt.grid(True, alpha=0.3)
plt.show()

summary = {
    'device': device,
    'dataset_path': str(Path(DEFAULT_DATASET_PATH).resolve()),
    'checkpoint_path': str(pinn_ckpt.resolve()),
    'best_epoch': best_epoch,
    'best_val_position_rmse_km': best_rmse,
    'model_config': model_cfg.to_kwargs(),
    'train_config': vars(train_cfg),
}
summary_path = pinn_ckpt.parent / 'pinn_train_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Training summary saved to:', summary_path)



Dataset source: /Users/francescosulli/Documents/ASTREO/IA e simulazioni/solar_system/PINN/data/de440.bsp
States shape: (58441, 10, 6)
Num samples: 58441


Training: 100%|██████████| 450/450 [15:52<00:00,  2.12s/it, lr=1.00e-06, nbody=0.000e+00, nbody_w=0.00e+00, phys=0.000e+00, phys_w=0.00e+00, pos_rmse_km=3.312e+05, smooth=0.000e+00, train=9.902e-07, val=1.008e-06]


Coarse checkpoint: /Users/francescosulli/Documents/ASTREO/IA e simulazioni/solar_system/PINN/artifacts/emulator_pinn_coarse.pt
Coarse best epoch: 449
Coarse best val position RMSE [km]: 330,352.95


Training: 100%|██████████| 220/220 [1:16:02<00:00, 20.74s/it, lr=1.00e-06, nbody=0.000e+00, nbody_w=0.00e+00, phys=0.000e+00, phys_w=0.00e+00, pos_rmse_km=3.142e+05, smooth=0.000e+00, train=1.170e-03, val=1.178e-03]


Refine checkpoint: /Users/francescosulli/Documents/ASTREO/IA e simulazioni/solar_system/PINN/artifacts/emulator_pinn_refine.pt
Refine best epoch: 216
Refine best val position RMSE [km]: 304,360.98


Training:  27%|██▋       | 16/60 [4:35:42<19:54:09, 1628.40s/it, lr=1.78e-06, nbody=5.442e-05, nbody_w=5.37e-05, phys=0.000e+00, phys_w=0.00e+00, pos_rmse_km=1.592e+05, smooth=0.000e+00, train=1.196e-06, val=1.191e-06]